# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a structured guide for loading, exploring, and processing a FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

The dataset contains survey data on mental health indicators among residents of Kilifi County, including demographic details and scores from assessments such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset metadata summary
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("License:", metadata.license)
print("Identifier:", metadata.identifier)
print("Keywords:", getattr(metadata, 'keywords', []))
print("Date Published:", metadata.datePublished)

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List all record sets (by @id)
record_sets = dataset.metadata.recordSet

print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'No name')} | {rs.get('description', 'No description')}")

# For each record set, show fields and columns referenced by @id
fields_dict = {}
columns_dict = {}
for rs in record_sets:
    print("\nRecord Set:", rs['@id'])
    if 'field' in rs:
        fields = rs['field']
        print("Fields:")
        fields_dict[rs['@id']] = []
        for f in fields:
            field_id = f['@id']
            label = f.get('name', f.get('label', ''))
            print(f"  - {field_id}: {label}")
            fields_dict[rs['@id']].append(field_id)
    if 'column' in rs:
        columns = rs['column']
        print("Columns:")
        columns_dict[rs['@id']] = []
        for c in columns:
            column_id = c['@id']
            label = c.get('name', c.get('label', ''))
            print(f"  - {column_id}: {label}")
            columns_dict[rs['@id']].append(column_id)
    print("---")

## 3. Data Extraction

Load data from one or more record sets into DataFrames for analysis. Record sets, fields, and columns are referenced strictly by their `@id`s.

In [ ]:
# Prepare to extract data from all record sets found above
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print("Columns:", df.columns.tolist())
        print(df.head())
    else:
        print("No records found.")

# Example: Choose the first record set for further processing
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    df_main = dataframes[main_record_set_id]
    print(f"\nSelected RecordSet for further analysis: {main_record_set_id}")
    print(df_main.head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping data. Reference fields and columns strictly by their `@id`.

In [ ]:
import numpy as np

# Example: Find a numeric field to analyze
numeric_field_id = None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Attempt to automatically select a numeric field (e.g., PHQ-9 score or GAD-7 score)
    for col in df.columns:
        if df[col].dtype in [np.float64, np.int64, float, int]:
            numeric_field_id = col
            break

    # If nothing found, try manually selecting one
    if numeric_field_id is None and len(df.columns) > 0:
        # Pick a likely numeric field by name
        for trial in ['phq9_score', 'gad7_score', 'psq_score', 'PHQ-9', 'GAD-7', 'PSQ']:
            for col in df.columns:
                if trial.lower() in col.lower():
                    numeric_field_id = col
                    break
            if numeric_field_id:
                break
    if not numeric_field_id:
        numeric_field_id = df.columns[0]

    print(f"Numeric field selected for analysis: {numeric_field_id}")
    
    # Filtering records based on a threshold
    threshold = 10
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalizing the numeric field
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, field_norm]].head())
        
        # Grouping by a categorical field if exists (e.g., gender or education)
        possible_groups = ['gender', 'level_of_education', 'village', 'marital_status']
        group_field_id = None
        for candidate in possible_groups:
            for col in df.columns:
                if candidate.lower() in col.lower():
                    group_field_id = col
                    break
            if group_field_id:
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id, field_norm].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Visualize group differences if group_field_id available
if main_record_set_id and numeric_field_id and 'group_field_id' in locals() and group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} scores by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- Successfully loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.
- Dataset provides valuable demographic and mental health data, with fields referenced by `@id` for robust standards.
- Exploratory analysis highlighted key numeric indicators with potential for further study, e.g., PHQ-9/GAD-7 distributions and relationships to demographic groups.
- This notebook can be extended for additional analyses, modeling, or data quality checks leveraging specific `@id` references.